# Avito Antibot — Bot Detection Challenge

Итоговое решение: классификация кук на ботов (сервисы сбора данных) и людей по событиям в суточном окне.

**Pipeline:** очистка событий → признаки внутри окна → time-based валидация → blend (HGB × 10 seed + ExtraTrees, 2:1) → `submission.csv`.

**Как запустить.** Положите `train.csv`, `test.csv`, `events.csv.gz` в папку `data/` рядом с ноутбуком (`metric.py` лежит в корне репозитория) и выполните ноутбук сверху вниз. Python 3.12.0, версии библиотек в `requirements.txt`.

**Структура ноутбука:** 1) данные и окружение → 2) очистка событий → 3) признаки → 4) валидация, baseline и модель → 5) проверка признаков → 6) submission → 7) воспроизводимость → описание решения.

## 1. Данные и окружение

In [14]:
import hashlib
import importlib.metadata as md
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import rankdata
from sklearn.ensemble import (ExtraTreesClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.metrics import average_precision_score, roc_auc_score
from threadpoolctl import threadpool_limits

from metric import precision_at_recall   # официальная метрика организаторов (metric.py в корне репозитория)

# Фиксируем все источники случайности, чтобы результат воспроизводился.
# Сами модели получают random_state явно (см. раздел 4); глобальные seed заданы на всякий случай.
SEED = 0
random.seed(SEED)
np.random.seed(SEED)

# Версии для requirements.txt и описания решения
print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("sklearn:", sklearn.__version__)

DATA_DIR = "data"

# Временные фолды (lo, hi): учимся на окнах до lo, проверяем на окнах [lo, hi).
# Тест лежит по времени после трейна, поэтому случайный K-fold дал бы завышенную оценку.
FOLDS = [("2026-04-11", "2026-04-14"),
         ("2026-04-14", "2026-04-17"),
         ("2026-04-17", "2026-04-20")]

# Даты парсим сразу: дальше по ним фильтруем события в окне наблюдения и строим фолды
train = pd.read_csv(f"{DATA_DIR}/train.csv",
                    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
test = pd.read_csv(f"{DATA_DIR}/test.csv",
                   parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
events = pd.read_csv(f"{DATA_DIR}/events.csv.gz", parse_dates=["event_ts"])

print("\ntrain:", train.shape, "| test:", test.shape, "| events:", events.shape)
print("доля ботов в train:", round(train.target.mean(), 4))
print("train окна:", train.window_start_ts.min().date(), "->", train.window_start_ts.max().date())
print("test окна: ", test.window_start_ts.min().date(), "->", test.window_start_ts.max().date())

Python : 3.12.0
pandas : 3.0.6
numpy  : 2.5.3
sklearn: 1.9.1

train: (11091, 5) | test: (4909, 4) | events: (328905, 14)
доля ботов в train: 0.0811
train окна: 2026-04-06 -> 2026-04-19
test окна:  2026-04-20 -> 2026-04-26


## 2. Очистка событий и окно наблюдения

Удаляем точные дубликаты, нормализуем `platform`, сортируем события и оставляем только те, что попали в окно `[window_start_ts, window_end_ts)`.

In [15]:
# Нормализация platform: склеиваем алиасы одной и той же платформы (регистр приводится к нижнему отдельно)
PLATFORM_MAP = {"desktop": "web", "iphone": "ios"}

def clean_events(events, drop_dups=True):
    """Чистим события до построения признаков.
    1. Точные дубликаты строк удаляем (drop_dups=True): доля дублей одинакова у ботов и людей, это шум.
    2. platform приводим к нижнему регистру и склеиваем алиасы.
    3. Сортируем по (cookie_id, event_ts); mergesort стабилен, поэтому результат детерминирован."""
    ev = events.drop_duplicates().copy() if drop_dups else events.copy()
    ev["platform"] = ev.platform.str.lower().replace(PLATFORM_MAP)
    return ev.sort_values(["cookie_id", "event_ts"], kind="mergesort").reset_index(drop=True)


def events_in_window(events, meta):
    """Признаки считаем только 
    по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
    Это требование из условия, а не рекомендация."""
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]


# Окна train и test чистятся одинаково; признаки для них строятся раздельно, но одним кодом
events_clean = clean_events(events)
ev_tr = events_in_window(events_clean, train)
ev_te = events_in_window(events_clean, test)

print("событий после дедупликации:", len(events_clean), "(было", len(events), ")")
print("в окне: train =", len(ev_tr), "| test =", len(ev_te))
print("platform:", events_clean.platform.value_counts().to_dict())
print("отсортировано внутри кук:",
      ev_tr.groupby("cookie_id").event_ts.apply(lambda s: s.is_monotonic_increasing).all())

событий после дедупликации: 324042 (было 328905 )
в окне: train = 195484 | test = 88349
platform: {'web': 182943, 'android': 124988, 'ios': 16111}
отсортировано внутри кук: True


## 3. Признаки

Группы признаков: базовая активность и состав событий, тайминг, User-Agent, курсор, популярность просматриваемого контента. Все строятся только по событиям внутри окна, таргет не используется.

In [16]:
# Фиксированный список типов событий: колонки признаков одинаковы для train и test
EVENT_NAMES = ["search_results_view", "item_view", "photo_swipe", "seller_page_view",
               "contact_phone_show", "contact_chat_open", "contact_message_sent",
               "favorite_add", "login", "captcha_shown"]
CONTACT = ["contact_phone_show", "contact_chat_open", "contact_message_sent"]

def group_entropy(ev, col):
    """Энтропия Шеннона распределения значений col внутри каждой куки."""
    c = ev.groupby(["cookie_id", col]).size()
    p = c / c.groupby(level=0).transform("sum")
    return -(p * np.log(p)).groupby(level=0).sum()

def build_base_features(ev, meta):
    """ev: очищенные события ВНУТРИ окна, отсортированные по (cookie_id, event_ts).
    meta: train/test. Возвращает одну строку на cookie_id из meta."""
    ev = ev.copy()
    g = ev.groupby("cookie_id")
    # Индекс = все куки из meta: у кук без событий в окне признаки останутся NaN
    # (HGB работает с NaN нативно, для ExtraTrees NaN заполняются в fit_predict_blend)
    F = pd.DataFrame(index=pd.Index(meta.cookie_id, name="cookie_id"))
    n = g.size()
    F["n_events"] = n

    # 1. Состав активности: счётчики и доли типов событий, энтропия типов
    cnt = pd.crosstab(ev.cookie_id, ev.event_name).reindex(columns=EVENT_NAMES, fill_value=0)
    for c in EVENT_NAMES:
        F[f"cnt_{c}"] = cnt[c]
        F[f"frac_{c}"] = cnt[c] / n
    p = cnt.div(n, axis=0).values   # доли типов событий; нулевые доли не вносят вклад в энтропию
    F["event_type_entropy"] = pd.Series(
        -np.where(p > 0, p * np.log(np.where(p > 0, p, 1)), 0).sum(axis=1), index=cnt.index)

    # 2. Тайминг: интервалы между соседними событиями куки (в секундах)
    # (события отсортированы, поэтому diff() внутри куки даёт интервал до предыдущего события)
    ev["gap"] = g.event_ts.diff().dt.total_seconds()
    ev["gap_lt1"] = (ev.gap < 1).where(ev.gap.notna())   # NaN у первого события куки не считаем
    ev["gap_lt2"] = (ev.gap < 2).where(ev.gap.notna())
    g = ev.groupby("cookie_id")   # пересоздаём groupby: в ev появились новые колонки
    F["session_span_sec"] = (g.event_ts.max() - g.event_ts.min()).dt.total_seconds()
    F["events_per_min"] = n / (F.session_span_sec / 60 + 1)
    F["gap_median"], F["gap_min"], F["gap_mean"] = g.gap.median(), g.gap.min(), g.gap.mean()
    F["gap_std"] = g.gap.std(ddof=0)
    F["gap_cv"] = F.gap_std / (F.gap_mean + 1e-6)   # коэффициент вариации: у скриптов интервалы более регулярны
    F["frac_gap_lt1s"], F["frac_gap_lt2s"] = g.gap_lt1.mean(), g.gap_lt2.mean()

    # 3. Разнообразие того, что смотрит кука
    for col, name in [("item_id", "items"), ("item_category", "categories"),
                      ("item_location", "locations"), ("search_query", "search_queries")]:
        F[f"n_unique_{name}"] = g[col].nunique()
    # доля повторных просмотров: 1 - (уникальные объявления / все просмотры с item_id)
    F["item_view_repeat_rate"] = 1 - F.n_unique_items / g.item_id.count().replace(0, np.nan)
    F["category_entropy"] = group_entropy(ev.dropna(subset=["item_category"]), "item_category")

    # 4. Технические признаки клиента и курсор
    F["n_unique_user_agent"] = g.user_agent.nunique()
    F["is_headless_frac"] = ev.user_agent.str.contains("Headless", case=False, na=False).groupby(ev.cookie_id).mean()
    for pl in ["web", "android", "ios"]:
        F[f"platform_{pl}_frac"] = (ev.platform == pl).groupby(ev.cookie_id).mean()
    F["pointer_present_frac"] = ev.pointer_x.notna().groupby(ev.cookie_id).mean()
    F["pointer_x_std"], F["pointer_y_std"] = g.pointer_x.std(), g.pointer_y.std()

    # 5. Конверсия и целевые действия (+1 в знаменателе защищает от деления на ноль)
    F["contact_rate"] = cnt[CONTACT].sum(axis=1) / (cnt.item_view + 1)
    F["favorite_rate"] = cnt.favorite_add / (cnt.item_view + 1)
    pro = (ev.seller_type == "pro").groupby(ev.cookie_id).sum()
    F["pro_seller_share"] = pro / g.seller_type.count().replace(0, np.nan)
    F["max_search_page"], F["mean_search_page"] = g.search_page.max(), g.search_page.mean()
    F["has_login"], F["has_captcha"] = (cnt.login > 0).astype(float), (cnt.captcha_shown > 0).astype(float)

    # 6. Время суток
    h = ev.event_ts.dt.hour   # часовой пояс в данных не указан, используем час как есть
    F["hour_mean"], F["hour_std"] = h.groupby(ev.cookie_id).mean(), h.groupby(ev.cookie_id).std(ddof=0)
    F["night_frac"] = (h < 7).groupby(ev.cookie_id).mean()

    # 7. Возраст куки на начало окна (в днях)
    m = meta.set_index("cookie_id")
    F["cookie_age_days"] = (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 86400
    return F.reset_index()

X_tr = build_base_features(ev_tr, train)
print(X_tr.shape, "| колонок с пропусками:", X_tr.isna().any().sum())

(11091, 57) | колонок с пропусками: 14


In [17]:
SESSION_BREAK_SEC = 30 * 60   # пауза больше 30 минут = новая "сессия"

def build_timing_features(ev, meta):
    """Тонкий тайминг: квантили интервалов, всплесковость, сессии, часы активности."""
    ev = ev.copy()   # не портим входной датафрейм
    ev["gap"] = ev.groupby("cookie_id").event_ts.diff().dt.total_seconds()
    ev["gap_known"] = ev.gap.notna()
    ev["is_break"] = ev.gap > SESSION_BREAK_SEC
    ev["hour"] = ev.event_ts.dt.hour
    ev["minute_of_day"] = ev.hour * 60 + ev.event_ts.dt.minute
    g = ev.groupby("cookie_id")
    F = pd.DataFrame(index=pd.Index(meta.cookie_id, name="cookie_id"))
    for q in (0.1, 0.25, 0.75, 0.9):
        F[f"gap_q{int(q * 100)}"] = g.gap.quantile(q)
    F["gap_max"] = g.gap.max()
    F["gap_logmean"] = np.log1p(ev.gap).groupby(ev.cookie_id).mean()
    m, s = g.gap.mean(), g.gap.std(ddof=0)
    F["gap_burstiness"] = (s - m) / (s + m + 1e-6)   # -1 = метроном, 0 = случайно, +1 = пачками
    F["frac_gap_eq0"] = (ev.gap == 0).where(ev.gap_known).groupby(ev.cookie_id).mean()
    F["frac_gap_le60"] = (ev.gap <= 60).where(ev.gap_known).groupby(ev.cookie_id).mean()
    F["n_sessions"] = g.is_break.sum() + 1
    F["events_per_session"] = g.size() / F.n_sessions
    F["n_unique_hours"] = g.hour.nunique()
    F["first_hour"], F["last_hour"] = g.hour.first(), g.hour.last()
    F["unique_minutes_ratio"] = g.minute_of_day.nunique() / g.size()
    return F.reset_index()

In [18]:
UA_FAMILIES = ["HeadlessChrome", "YaBrowser", "Edg", "Firefox", "Chrome", "Safari"]  # порядок важен: первое совпадение
UA_OS = ["Windows", "Macintosh", "X11", "Android", "iPhone"]

def parse_ua(ua):
    """Разбор строки UA в числовые признаки (сама строка модели не подходит: 148 уникальных значений)."""
    family = next((i for i, k in enumerate(UA_FAMILIES) if k in ua), len(UA_FAMILIES))
    os_ = next((i for i, k in enumerate(UA_OS) if k in ua), len(UA_OS))
    v = re.search(r"(?:Chrome|Firefox)/(\d+)", ua)
    yb = re.search(r"YaBrowser/(\d+)\.(\d+)", ua)
    return pd.Series({"ua_family": family, "ua_os": os_,
                      "ua_major_ver": float(v.group(1)) if v else np.nan,
                      "ua_yandex_ver": float(yb.group(1)) * 10 + float(yb.group(2)) if yb else np.nan})

def build_ua_features(ev, meta):
    """У куки в среднем ~1.1 UA, берём самый частый и разбираем его один раз на уникальное значение."""
    mode_ua = ev.groupby("cookie_id").user_agent.agg(lambda s: s.mode().iloc[0])
    uniq = pd.Series(mode_ua.unique())
    table = pd.concat([uniq.rename("ua"), uniq.apply(parse_ua)], axis=1).set_index("ua")
    F = table.loc[mode_ua.values].set_index(mode_ua.index)
    return meta[["cookie_id"]].merge(F.reset_index(), on="cookie_id", how="left")

In [19]:
# Размер экрана принят константой (1920x1080): для мобильных платформ это лишь приближение
SCREEN_W, SCREEN_H = 1920, 1080

def build_pointer_features(ev, meta):
    """У людей координаты разбросаны по экрану равномерно, у ботов сдвинуты и часто прижаты к краям."""
    p = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()
    g = p.groupby("cookie_id")
    F = pd.DataFrame(index=pd.Index(meta.cookie_id, name="cookie_id"))
    F["ptr_n"] = g.size()
    for c, a in (("pointer_x", "x"), ("pointer_y", "y")):
        F[f"ptr_{a}_mean"], F[f"ptr_{a}_min"], F[f"ptr_{a}_max"] = g[c].mean(), g[c].min(), g[c].max()
    p["x_lo"], p["x_hi"] = p.pointer_x <= 0, p.pointer_x >= SCREEN_W - 1
    p["y_lo"], p["y_hi"] = p.pointer_y <= 0, p.pointer_y >= SCREEN_H - 1
    p["on_edge"] = p.x_lo | p.x_hi | p.y_lo | p.y_hi
    for c in ("x_lo", "x_hi", "y_lo", "y_hi", "on_edge"):
        F[f"ptr_{c}_frac"] = p.groupby("cookie_id")[c].mean()
    step = np.hypot(g.pointer_x.diff().abs(), g.pointer_y.diff().abs())   # расстояние между соседними точками
    F["ptr_step_mean"] = step.groupby(p.cookie_id).mean()
    key = p.pointer_x * 10_000 + p.pointer_y   # пара (x, y) -> одно число, чтобы считать уникальные позиции
    F["ptr_unique_ratio"] = key.groupby(p.cookie_id).nunique() / F.ptr_n
    return F.reset_index()

In [20]:
LOOKBACK_DAYS = 6   # текущий день + 6 предыдущих

def build_popularity_features(ev_all, ev, meta, lookback=LOOKBACK_DAYS):
    """Сколько ДРУГИХ кук смотрели те же объявления/запросы в днях [D-lookback, D],
    где D — день окна самой куки; более поздние дни не используются.
    Таргет не задействован; ev_all — события в окнах всех кук train+test
    (признак неконтролируемый, но использует события других кук, в том числе тестовых)."""
    ea = ev_all.assign(day=ev_all.window_start_ts.dt.normalize())
    days = pd.date_range(ea.day.min(), ea.day.max())
    x0 = ev.assign(day=ev.window_start_ts.dt.normalize())
    F = pd.DataFrame(index=pd.Index(meta.cookie_id, name="cookie_id"))
    for col, name in (("item_id", "item"), ("search_query", "query")):
        # уникальные куки на (объект, день); каждая кука живёт ровно один день, поэтому суммы по дням корректны
        per_day = ea.dropna(subset=[col]).groupby([col, "day"]).cookie_id.nunique().unstack(fill_value=0)
        per_day = per_day.reindex(columns=days, fill_value=0)
        # скользящая сумма по дням: текущий день + lookback предыдущих (для первых дней окно короче)
        window_sum = per_day.T.rolling(lookback + 1, min_periods=1).sum().T
        x = x0.dropna(subset=[col])[["cookie_id", col, "day"]].copy()
        cell = window_sum.reindex(x[col]).values[np.arange(len(x)), days.get_indexer(x["day"])]
        x["pop"] = cell - 1   # вычитаем саму куку, её вклад в счётчик всегда равен 1
        g = x.groupby("cookie_id")["pop"]
        F[f"{name}_pop_mean"], F[f"{name}_pop_max"] = g.mean(), g.max()
        F[f"{name}_pop_pos"] = (x["pop"] > 0).groupby(x.cookie_id).mean()
    return F.reset_index()

In [21]:
ev_all = pd.concat([ev_tr, ev_te], ignore_index=True)   # события в окнах всех кук train+test, без таргета


def build_all_features(ev, meta):
    """Собирает все группы признаков в одну таблицу (одна строка на cookie_id из meta).
    Порядок колонок одинаков для train и test; ev_all берётся из глобальной области."""
    X = build_base_features(ev, meta)
    for G in (
        build_timing_features(ev, meta),
        build_ua_features(ev, meta),
        build_pointer_features(ev, meta),
        build_popularity_features(ev_all, ev, meta),
    ):
        X = X.merge(G, on="cookie_id", how="left")
    return X


X_tr = build_all_features(ev_tr, train)
X_te = build_all_features(ev_te, test)

print(X_tr.shape, X_te.shape)


(11091, 96) (4909, 96)


## 4. Валидация, baseline и модель

Темпоральная CV с официальной метрикой. Ниже сначала финальный blend, затем baseline из quickstart на тех же фолдах.

In [22]:
FEATURE_COLS = [c for c in X_tr.columns if c != "cookie_id"]

# Параметры HGB: неглубокие деревья + маленький шаг + L2 (данных мало, ботов ~8%)
HGB_BLEND_PARAMS = dict(
    max_iter=600,
    learning_rate=0.03,
    max_depth=4,
    l2_regularization=2.0,
    max_features=0.5,
    early_stopping=False,
)
N_SEEDS = 10          # число seed для усреднения HGB (seed влияет через max_features < 1)
W_HGB, W_ET = 2, 1    # веса blend, подобраны вручную

def to_rank(p):
    """Преобразуем вероятности в ранги; метрика зависит только от порядка score."""
    return rankdata(p) / len(p)


def fit_predict_blend(X_fit, y_fit, X_pred):
    """Детерминированный blend: HGB (10 seed) + ExtraTrees, веса 2:1."""
    # Один поток: порядок суммирования в многопоточных расчётах может слегка менять результат
    with threadpool_limits(limits=1):
        hgb_p = np.mean([
            HistGradientBoostingClassifier(
                random_state=SEED + s,
                **HGB_BLEND_PARAMS,
            )
            .fit(X_fit, y_fit)
            .predict_proba(X_pred)[:, 1]
            for s in range(N_SEEDS)
        ], axis=0)

        # ExtraTrees не принимает NaN, поэтому пропуски заменяем на -1 (значение вне диапазона признаков)
        et = ExtraTreesClassifier(
            n_estimators=600,
            min_samples_leaf=2,
            max_features=0.4,
            n_jobs=1,
            random_state=SEED,
        )
        et_p = (
            et.fit(X_fit.fillna(-1), y_fit)
            .predict_proba(X_pred.fillna(-1))[:, 1]
        )

    # Смешиваем ранги, а не вероятности: у моделей разная калибровка, а метрика зависит только от порядка
    return (W_HGB * to_rank(hgb_p) + W_ET * to_rank(et_p)) / (W_HGB + W_ET)


def run_cv_blend(X, meta):
    """Временная CV с официальной метрикой challenge.
    Для каждого фолда: обучение на окнах строго до lo, проверка на окнах [lo, hi).
    Признаки строятся один раз для всех окон и не зависят от границ фолда."""
    d = meta.set_index("cookie_id").loc[X.cookie_id]
    y, ws = d.target.values, d.window_start_ts
    res = []
    for lo, hi in FOLDS:
        tr = (ws < lo).values                        # только прошлое относительно валидации
        va = ((ws >= lo) & (ws < hi)).values
        p = fit_predict_blend(X.loc[tr, FEATURE_COLS], y[tr], X.loc[va, FEATURE_COLS])
        res.append((
            precision_at_recall(y[va], p),
            average_precision_score(y[va], p),
            roc_auc_score(y[va], p),
            va.sum(),
            y[va].sum(),
        ))

    r = pd.DataFrame(
        res,
        columns=["P@R0.7", "PR-AUC", "ROC-AUC", "n_valid", "n_bots"],
        index=[f"valid {lo}..{hi}" for lo, hi in FOLDS],
    )
    print(r.round(4))
    print(
        "mean P@R0.7 =", round(r["P@R0.7"].mean(), 4),
        "| mean PR-AUC =", round(r["PR-AUC"].mean(), 4),
    )
    return r


cv_blend = run_cv_blend(X_tr, train)


                              P@R0.7  PR-AUC  ROC-AUC  n_valid  n_bots
valid 2026-04-11..2026-04-14  0.7692  0.7867   0.9378     2556     199
valid 2026-04-14..2026-04-17  0.8107  0.8261   0.9566     2367     195
valid 2026-04-17..2026-04-20  0.8485  0.8113   0.9458     1951     160
mean P@R0.7 = 0.8095 | mean PR-AUC = 0.808


In [23]:
# Baseline из quickstart на тех же временных фолдах, что и финальный blend.
# Признаки: только n_events и n_unique_items; модель: RandomForest с параметрами из quickstart.
BASELINE_COLS = ["n_events", "n_unique_items"]

d_b = train.set_index("cookie_id").loc[X_tr.cookie_id]
y_b, ws_b = d_b.target.values, d_b.window_start_ts
Xb = X_tr[BASELINE_COLS].fillna(0)   # как в quickstart: нет событий в окне -> 0

res_b = []
for lo, hi in FOLDS:
    tr = (ws_b < lo).values                       # те же правила разбиения, что в run_cv_blend
    va = ((ws_b >= lo) & (ws_b < hi)).values
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                random_state=SEED, n_jobs=1)
    p = rf.fit(Xb.loc[tr], y_b[tr]).predict_proba(Xb.loc[va])[:, 1]
    res_b.append((precision_at_recall(y_b[va], p),
                  average_precision_score(y_b[va], p),
                  roc_auc_score(y_b[va], p)))

cv_baseline = pd.DataFrame(res_b, columns=["P@R0.7", "PR-AUC", "ROC-AUC"],
                           index=[f"valid {lo}..{hi}" for lo, hi in FOLDS])
print(cv_baseline.round(4))
print("baseline mean P@R0.7 =", round(cv_baseline["P@R0.7"].mean(), 4),
      "| final blend mean P@R0.7 =", round(cv_blend["P@R0.7"].mean(), 4))

                              P@R0.7  PR-AUC  ROC-AUC
valid 2026-04-11..2026-04-14  0.0932  0.2318   0.6252
valid 2026-04-14..2026-04-17  0.0996  0.2295   0.6207
valid 2026-04-17..2026-04-20  0.0957  0.1634   0.5724
baseline mean P@R0.7 = 0.0962 | final blend mean P@R0.7 = 0.8095


## 5. Проверка финальных признаков

В финальной ветке используются 95 числовых признаков. Ни один target-признак не строится.

In [24]:
# Согласованность таблиц признаков: одинаковые колонки, по одной строке на куку, полное покрытие meta
assert list(X_tr.columns) == list(X_te.columns)
assert X_tr.cookie_id.is_unique and X_te.cookie_id.is_unique
assert set(X_tr.cookie_id) == set(train.cookie_id)
assert set(X_te.cookie_id) == set(test.cookie_id)
# Все признаки числовые; пропуски (куки без событий, отсутствие user_agent/курсора) допустимы
FEATURE_COLS = [c for c in X_tr.columns if c != "cookie_id"]
assert len(FEATURE_COLS) == 95, f"Ожидалось 95 признаков, получено {len(FEATURE_COLS)}"
print("X_tr:", X_tr.shape, "| X_te:", X_te.shape)
print("число признаков:", len(FEATURE_COLS))
print("NaN train:", int(X_tr[FEATURE_COLS].isna().sum().sum()))
print("NaN test :", int(X_te[FEATURE_COLS].isna().sum().sum()))


X_tr: (11091, 96) | X_te: (4909, 96)
число признаков: 95
NaN train: 135139
NaN test : 59268


## 6. Submission

Обучаем blend на всём train и предсказываем test.

In [25]:
# Финальное обучение на всём train (все окна), предсказание для test
y_all = train.set_index("cookie_id").loc[X_tr.cookie_id, "target"].values

score = fit_predict_blend(X_tr[FEATURE_COLS], y_all, X_te[FEATURE_COLS])
sub = pd.DataFrame({"cookie_id": X_te.cookie_id.values, "score": score})

# Проверка формата: две колонки, по одной строке на куку из test, score в [0, 1] без пропусков
assert list(sub.columns) == ["cookie_id", "score"]
assert len(sub) == len(test)
assert sub.cookie_id.is_unique
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.notna().all()
assert sub.score.between(0, 1).all()

sub.to_csv("submission.csv", index=False)

print("submission:", sub.shape)
print("score min/max:", round(sub.score.min(), 6), round(sub.score.max(), 6))
print(sub.head())


submission: (4909, 2)
score min/max: 0.004991 0.999389
             cookie_id     score
0  ck_315fb710a0e371e7  0.096218
1  ck_a76ee3b3e3e522fd  0.886331
2  ck_94c9a4d382689e82  0.633259
3  ck_8eaf9509ad9462a0  0.143274
4  ck_9a88a5a989cb5bc6  0.381272


## 7. Проверка воспроизводимости

Повторный запуск обучения должен дать побайтно тот же файл. Здесь же фиксируем версии библиотек в `requirements.txt`.

In [26]:
# Полный повторный fit на тех же данных: файл должен совпасть побайтно
def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

score_rerun = fit_predict_blend(X_tr[FEATURE_COLS], y_all, X_te[FEATURE_COLS])
sub_rerun = pd.DataFrame({"cookie_id": X_te.cookie_id.values, "score": score_rerun})
sub_rerun.to_csv("submission_rerun.csv", index=False)

print("md5 submission.csv      :", md5("submission.csv"))
print("md5 submission_rerun.csv:", md5("submission_rerun.csv"))
print("идентичны:", md5("submission.csv") == md5("submission_rerun.csv"))

assert np.array_equal(sub.score.values, sub_rerun.score.values)


md5 submission.csv      : 22bd7026709d5bf045a2444caa25be2f
md5 submission_rerun.csv: 22bd7026709d5bf045a2444caa25be2f
идентичны: True


In [27]:
# Версии установленных библиотек -> requirements.txt
packages = ["pandas", "numpy", "scikit-learn", "scipy", "threadpoolctl"]
requirements = [f"{pkg}=={md.version(pkg)}" for pkg in packages]
Path("requirements.txt").write_text("\n".join(requirements) + "\n", encoding="utf-8")

print("Python", sys.version.split()[0])
print("\n".join(requirements))


Python 3.12.0
pandas==3.0.6
numpy==2.5.3
scikit-learn==1.9.1
scipy==1.18.1
threadpoolctl==3.7.0


## Описание решения

**Данные.** Полные дубликаты событий удаляются; `platform` нормализуется (`desktop`→`web`, `iphone`→`ios`), события сортируются по `cookie_id, event_ts`. Признаки строятся только по событиям внутри окна `[window_start_ts, window_end_ts)`.

**Признаки (95).** Базовая активность и состав типов событий, тайминг (интервалы между событиями, квантили, всплесковость, сессии), разнообразие объявлений/категорий/локаций/запросов, User-Agent (семейство, ОС, версия; сырая строка не используется), координаты курсора, контактные действия, глубина поиска, капча, время суток, возраст куки и признаки популярности просматриваемых объявлений/запросов среди других кук за текущий и 6 предыдущих дней. Таргет при построении признаков не используется.

**Валидация.** Time-based схема из трёх фолдов: обучение на окнах строго до границы, проверка на следующих трёх днях (тест лежит позже трейна по времени). Основная метрика считается официальной функцией `precision_at_recall` из `metric.py`; дополнительно смотрим PR-AUC и ROC-AUC.

**Модель.** `HistGradientBoostingClassifier` усредняется по 10 seed, затем смешивается с `ExtraTreesClassifier` в пропорции 2:1. Перед смешиванием вероятности переводятся в ранги (метрика зависит только от порядка score). `early_stopping=False`, обучение однопоточное для воспроизводимости.

**Результаты.**

| Конфигурация | Средний P@R≥0.7 (CV, 3 фолда) |
|---|---|
| Baseline из quickstart (RandomForest на n_events, n_unique_items) | 0.0962 |
| Базовые признаки после очистки событий (HGB) | ≈ 0.699 |
| Все группы признаков (HGB) | ≈ 0.777 |
| Финальный blend HGB×10 seed + ExtraTrees | 0.8095 (вывод CV выше) |

Baseline (0.0962) и финальный blend (0.8095) считаются в этом ноутбуке на одних и тех же временных фолдах (раздел 4). Две промежуточные строки (≈ 0.699 и ≈ 0.777) взяты из экспериментального ноутбука `v1`, который в репозиторий не входит. Baseline лежит на уровне доли ботов (≈ 8%), то есть почти константа. На скрытом тесте итоговый **Precision@Recall≥0.70 = 0.88798** (по данным проверяющей системы).

**Ограничения.** В каждом фолде 160–199 ботов, поэтому P@R≥0.7 заметно шумит от фолда к фолду (0.77–0.85). Признаки популярности используют события других кук (в том числе тестовых) за текущий и предыдущие дни, но без таргета. Границы экрана для признаков курсора заданы константами 1920×1080. Blend-веса 2:1 зафиксированы вручную.

**Воспроизводимость.** Зафиксирован `SEED=0`, `early_stopping=False`, `n_jobs=1`, `threadpool_limits(1)`. Ячейка проверки повторно строит submission и сравнивает MD5 двух запусков. В разделе 7 записывается `requirements.txt` с версиями установленных библиотек.

**Submission.** Ровно `submission.csv` с колонками `cookie_id` и `score`; порог отправлять не нужно.

**Использованные инструменты.** pandas, numpy, scikit-learn (HistGradientBoosting, ExtraTrees, RandomForest для baseline), scipy (`rankdata`), threadpoolctl. Все библиотеки open-source и работают локально; большие языковые модели и внешние API не использовались.